# دانلود همهٔ ویدیوهای دوره سواد مالی

این نوت‌بوک ۴۱ فعالیت را از `course_page.html` می‌خواند، لینک پخش هر فعالیت را با نشست واردشدهٔ Chrome استخراج می‌کند و ویدیوها را در `data/raw_videos/` دانلود می‌کند.

> فقط محتوایی را دانلود کنید که مجاز به دسترسی و نگهداری آن هستید. فایل HTML، کوکی‌ها و ویدیوها نباید وارد Git شوند.

In [ ]:
from pathlib import Path
from urllib.parse import urljoin
from http.cookiejar import MozillaCookieJar
import html
import json
import re
import shutil
import subprocess
import tempfile

import requests
from bs4 import BeautifulSoup

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DOWNLOAD_DIR = PROJECT_ROOT / "data" / "raw_videos"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

yt_dlp = shutil.which("yt-dlp")
if yt_dlp is None:
    local_binary = Path.home() / "bin" / "yt-dlp"
    if local_binary.exists():
        yt_dlp = str(local_binary)
    else:
        raise FileNotFoundError("yt-dlp پیدا نشد. ابتدا آن را نصب کنید: python3 -m pip install -U yt-dlp")

print("Project root:", PROJECT_ROOT)
print("Download folder:", DOWNLOAD_DIR)

## ۱. خواندن همهٔ فعالیت‌ها

فایل `course_page.html` را در ریشهٔ پروژه یا کنار نوت‌بوک بگذارید.

In [ ]:
BASE_URL = "https://lms.fintelligence.ir"
html_candidates = [CWD / "course_page.html", PROJECT_ROOT / "course_page.html"]
HTML_FILE = next((path for path in html_candidates if path.is_file()), None)

if HTML_FILE is None:
    checked = "\n".join(f"- {path}" for path in html_candidates)
    raise FileNotFoundError(f"course_page.html پیدا نشد. مسیرهای بررسی‌شده:\n{checked}")

with HTML_FILE.open("r", encoding="utf-8", errors="ignore") as file:
    soup = BeautifulSoup(file, "html.parser")

activities = []
seen_urls = set()
for link in soup.select("a.activity-ajax"):
    href = link.get("href")
    if not href or href == "#":
        continue
    url = urljoin(BASE_URL, href)
    if url in seen_urls:
        continue
    seen_urls.add(url)
    activities.append({
        "id": link.get("data-id"),
        "title": " ".join(link.get_text(" ", strip=True).split()),
        "url": url,
    })

if not activities:
    raise RuntimeError("هیچ فعالیتی در فایل HTML پیدا نشد.")

print("Activity count:", len(activities))
for index, activity in enumerate(activities, 1):
    print(f"{index:03d} | {activity['title']}")

## ۲. ساخت نشست و استخراج لینک پخش

قبل از اجرا، دوره را در Chrome باز کنید و مطمئن شوید وارد حساب خود هستید. این سلول برای هر فعالیت پاسخ AJAX را می‌گیرد و لینک `m3u8` را پیدا می‌کند.

In [ ]:
cookie_file = Path(tempfile.gettempdir()) / "fintelligence_cookies.txt"
export_command = [
    yt_dlp,
    "--cookies-from-browser", "chrome",
    "--cookies", str(cookie_file),
    "--skip-download",
    activities[0]["url"],
]
subprocess.run(export_command, check=False, stdout=subprocess.DEVNULL)

if not cookie_file.exists():
    raise FileNotFoundError("کوکی Chrome استخراج نشد. Chrome را باز کنید، وارد سایت شوید و دوباره اجرا کنید.")

cookie_jar = MozillaCookieJar(str(cookie_file))
cookie_jar.load(ignore_discard=True, ignore_expires=True)

session = requests.Session()
session.cookies = cookie_jar
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/140.0.0.0 Safari/537.36",
    "Referer": f"{BASE_URL}/",
    "X-Requested-With": "XMLHttpRequest",
    "Accept": "application/json, text/javascript, */*; q=0.01",
})

M3U8_PATTERN = re.compile(r'https?://[^\s"\'<>\\]+?\.m3u8(?:\?[^\s"\'<>\\]*)?', re.IGNORECASE)
VIDEO_PATH_PATTERN = re.compile(r'(?:"|\')((?:https?://[^"\']+)?/video/play/[^"\']+?\.m3u8(?:\?[^"\']*)?)(?:"|\')', re.IGNORECASE)

def normalize_payload_text(value):
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False)
    text = html.unescape(str(value))
    text = text.replace("\\/", "/").replace("\\u0026", "&").replace("&amp;", "&")
    return text

def find_m3u8(value, base_url):
    text = normalize_payload_text(value)
    candidates = M3U8_PATTERN.findall(text)
    candidates += VIDEO_PATH_PATTERN.findall(text)
    candidates = [urljoin(base_url, candidate) for candidate in candidates]
    candidates = list(dict.fromkeys(candidates))
    if not candidates:
        return None
    return next((url for url in candidates if "master.m3u8" in url), candidates[0])

videos = []
failed_activities = []

for index, activity in enumerate(activities, 1):
    response = session.get(activity["url"], params={"ajax": 1}, timeout=45)
    if "/login" in response.url or "/user/session" in response.url:
        raise RuntimeError("نشست ورود معتبر نیست. دوره را در Chrome باز کنید و این سلول را دوباره اجرا کنید.")
    response.raise_for_status()

    try:
        payload = response.json()
    except ValueError:
        payload = response.text

    stream_url = find_m3u8(payload, response.url)
    if stream_url is None:
        failed_activities.append(activity)
        print(f"{index:03d} | NOT FOUND | {activity['title']}")
        continue

    video = {
        **activity,
        "name": f"session-{index:02d}",
        "stream_url": stream_url,
    }
    videos.append(video)
    print(f"{index:03d} | FOUND | {activity['title']}")

print(f"\nLinks found: {len(videos)} / {len(activities)}")
if failed_activities:
    print("Activities without a video link:")
    for item in failed_activities:
        print("-", item["url"])

links_file = DOWNLOAD_DIR / "video_links.json"
links_file.write_text(json.dumps(videos, ensure_ascii=False, indent=2), encoding="utf-8")
print("Local link manifest:", links_file)

## ۳. دانلود همهٔ ویدیوها

این سلول تمام لینک‌های پیدا‌شده را دانلود می‌کند. فایل MP4 موجود و غیرخالی دوباره دانلود نمی‌شود. خطای یک قسمت مانع ادامهٔ بقیه نمی‌شود.

In [ ]:
if not videos:
    raise RuntimeError("هیچ لینک ویدیویی استخراج نشده است؛ ابتدا سلول قبلی را بررسی کنید.")

download_errors = []
for index, video in enumerate(videos, 1):
    completed_file = DOWNLOAD_DIR / f"{video['name']}.mp4"
    if completed_file.is_file() and completed_file.stat().st_size > 0:
        print(f"SKIP {video['name']}: already downloaded")
        continue

    output_template = str(DOWNLOAD_DIR / f"{video['name']}.%(ext)s")
    command = [
        yt_dlp,
        "--cookies", str(cookie_file),
        "--referer", video["url"],
        "--merge-output-format", "mp4",
        "--continue",
        "--no-overwrites",
        "--retries", "10",
        "--fragment-retries", "10",
        "-o", output_template,
        video["stream_url"],
    ]
    print(f"\n[{index}/{len(videos)}] Downloading {video['name']} | {video['title']}")
    result = subprocess.run(command, check=False)
    if result.returncode != 0:
        download_errors.append(video)
        print(f"FAILED: {video['name']} (continuing)")

downloaded_files = sorted(DOWNLOAD_DIR.glob("session-*.mp4"))
print(f"\nDownloaded files: {len(downloaded_files)}")
print(f"Download errors: {len(download_errors)}")
for item in download_errors:
    print("-", item["name"], item["url"])

## ۴. بررسی نتیجه

In [ ]:
downloaded_files = sorted(DOWNLOAD_DIR.glob("session-*.mp4"))
expected_names = {video["name"] for video in videos}
downloaded_names = {path.stem for path in downloaded_files}
missing_names = sorted(expected_names - downloaded_names)

print("Activities:", len(activities))
print("Video links:", len(videos))
print("Downloaded:", len(downloaded_files))
print("Missing:", len(missing_names))
if missing_names:
    print("Missing sessions:", ", ".join(missing_names))

ffprobe = shutil.which("ffprobe")
if ffprobe is None:
    print("ffprobe نصب نیست؛ برای بررسی فنی در macOS اجرا کنید: brew install ffmpeg")
else:
    invalid_files = []
    for file_path in downloaded_files:
        result = subprocess.run(
            [ffprobe, "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=codec_name", "-of", "csv=p=0", str(file_path)],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0 or not result.stdout.strip():
            invalid_files.append(file_path.name)
    print("Invalid files:", len(invalid_files))
    for name in invalid_files:
        print("-", name)